# Improvement Study: Poincaré Ball vs MaxSim vs Baseline
**Project:** Semantic Similarity Measurement in Latent Space for LLM Prediction Evaluation  
**Part 4 — Failure Analysis and Improvement**

This notebook compares three similarity metrics:
- **Baseline**: Standard cosine similarity in Euclidean space
- **Poincaré Ball**: Hyperbolic distance after exponential-map projection
- **MaxSim**: Sentence-level maximum cosine similarity

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import roc_curve, auc as sklearn_auc

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)

RESULTS_DIR  = '../results'
FIGURES_DIR  = '../results/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

DATASETS   = ['sciq', 'simpleqa', 'natural_questions', 'truthfulqa']
MODES      = ['no_thinking', 'thinking']
EMB_MODELS = ['bge-base', 'minilm', 'e5-base']
SHORT_FORM = ['sciq', 'simpleqa']
LONG_FORM  = ['natural_questions', 'truthfulqa']
METHODS    = ['baseline', 'poincare', 'maxsim']

METHOD_COLORS = {
    'baseline': '#607D8B',
    'poincare': '#9C27B0',
    'maxsim':   '#FF5722',
}
PALETTE = {'correct': '#2196F3', 'incorrect': '#F44336'}

print('Imports OK')

## 1. Load Results

In [ ]:
# Load improvement summary (baseline + poincare + maxsim AUCs)
imp_path = os.path.join(RESULTS_DIR, 'improvement_summary.csv')
imp_df = pd.read_csv(imp_path) if os.path.exists(imp_path) else pd.DataFrame()
print(f'Loaded {len(imp_df)} rows from improvement_summary.csv')
imp_df.head(9)

In [ ]:
# Load per-sample similarity CSVs for all three methods
def load_similarity_csvs(method_prefix):
    """Load all per-sample CSVs for a given method prefix."""
    frames = []
    prefix = '' if method_prefix == 'baseline' else f'{method_prefix}_'
    for ds in DATASETS:
        for mode in MODES:
            for emb in EMB_MODELS:
                path = os.path.join(RESULTS_DIR, f'{prefix}{ds}_{mode}_{emb}_similarity.csv')
                if os.path.exists(path):
                    df = pd.read_csv(path)
                    df['emb_model'] = emb
                    df['method']    = method_prefix
                    frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

base_df     = load_similarity_csvs('baseline')
poincare_df = load_similarity_csvs('poincare')
maxsim_df   = load_similarity_csvs('maxsim')

all_df = pd.concat([base_df, poincare_df, maxsim_df], ignore_index=True)
print(f'baseline: {len(base_df)} rows')
print(f'poincare: {len(poincare_df)} rows')
print(f'maxsim:   {len(maxsim_df)} rows')

## 2. AUC Comparison — Grouped Bar Chart

In [ ]:
def plot_auc_comparison(imp_df, emb='bge-base', mode='no_thinking', save=True):
    """Side-by-side grouped bar chart: Baseline / Poincaré / MaxSim × dataset."""
    sub = imp_df[(imp_df['emb_model'] == emb) & (imp_df['thinking_mode'] == mode)].copy()
    if sub.empty:
        print('No data')
        return

    available_methods = [m for m in METHODS if m in sub['method'].unique()]
    pivot = sub.pivot(index='dataset', columns='method', values='auc')
    pivot = pivot.reindex(
        index=[d for d in DATASETS if d in pivot.index],
        columns=[m for m in METHODS if m in pivot.columns]
    )

    x = np.arange(len(pivot))
    width = 0.25
    fig, ax = plt.subplots(figsize=(12, 5))

    for i, method in enumerate(available_methods):
        if method not in pivot.columns:
            continue
        offset = (i - len(available_methods)/2 + 0.5) * width
        bars = ax.bar(x + offset, pivot[method], width,
                      label=method.capitalize(),
                      color=METHOD_COLORS[method], alpha=0.85, edgecolor='white')
        # Add value labels
        for bar in bars:
            h = bar.get_height()
            if not np.isnan(h):
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=7)

    ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8, label='Random')
    ax.set_xticks(x)
    ax.set_xticklabels([
        f'{d}\n({"short" if d in SHORT_FORM else "long"})' for d in pivot.index
    ])
    ax.set_ylim([0.3, 1.05])
    ax.set_ylabel('AUC')
    ax.set_title(f'AUC: Baseline vs Poincaré vs MaxSim  |  emb={emb}  |  mode={mode}',
                 fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    if save:
        path = os.path.join(FIGURES_DIR, f'auc_comparison_{mode}_{emb}.pdf')
        fig.savefig(path, bbox_inches='tight')
        print(f'Saved → {path}')
    plt.show()

for mode in MODES:
    plot_auc_comparison(imp_df, emb='bge-base', mode=mode)

## 3. ΔAUC Heatmap (vs Baseline)

In [ ]:
def plot_delta_auc_heatmap(imp_df, method='poincare', mode='no_thinking', save=True):
    """Heatmap of ΔAUC = method_AUC − baseline_AUC across (dataset × emb_model)."""
    sub = imp_df[imp_df['thinking_mode'] == mode]
    baseline = sub[sub['method'] == 'baseline'][['dataset','emb_model','auc']]
    improved = sub[sub['method'] == method][['dataset','emb_model','auc']]
    if baseline.empty or improved.empty:
        print(f'Missing data for {method}')
        return

    merged = improved.merge(baseline, on=['dataset','emb_model'], suffixes=('_imp','_base'))
    merged['delta'] = merged['auc_imp'] - merged['auc_base']

    pivot = merged.pivot(index='dataset', columns='emb_model', values='delta')
    pivot = pivot.reindex(
        index=[d for d in DATASETS if d in pivot.index],
        columns=[e for e in EMB_MODELS if e in pivot.columns]
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    vmax = max(abs(pivot.values[~np.isnan(pivot.values)]).max(), 0.01)
    sns.heatmap(
        pivot, annot=True, fmt='+.4f', cmap='RdYlGn',
        center=0, vmin=-vmax, vmax=vmax,
        linewidths=0.5, ax=ax, cbar_kws={'label': 'ΔAUC'}
    )
    ax.set_title(f'ΔAUC ({method} − baseline)  |  mode={mode}', fontweight='bold')
    ax.set_xlabel('Embedding Model')
    ax.set_ylabel('Dataset')
    plt.tight_layout()
    if save:
        path = os.path.join(FIGURES_DIR, f'delta_auc_{method}_{mode}.pdf')
        fig.savefig(path, bbox_inches='tight')
        print(f'Saved → {path}')
    plt.show()

for method in ['poincare', 'maxsim']:
    for mode in MODES:
        plot_delta_auc_heatmap(imp_df, method=method, mode=mode)

## 4. ROC Curve Overlay — TruthfulQA (Focus Dataset)

In [ ]:
def plot_roc_overlay(all_df, dataset='truthfulqa', emb='bge-base', save=True):
    """Three-method ROC overlay for a specific dataset."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'ROC Curve Overlay — {dataset}  |  emb={emb}', fontweight='bold')

    line_styles = {'baseline': '-', 'poincare': '--', 'maxsim': ':'}

    for ax, mode in zip(axes, MODES):
        sub = all_df[
            (all_df['dataset'] == dataset) &
            (all_df['thinking_mode'] == mode) &
            (all_df['emb_model'] == emb)
        ]
        for method in METHODS:
            mdata = sub[sub['method'] == method]
            if mdata.empty or mdata['is_correct'].nunique() < 2:
                continue
            fpr, tpr, _ = roc_curve(mdata['is_correct'], mdata['similarity'])
            roc_auc = sklearn_auc(fpr, tpr)
            ax.plot(fpr, tpr,
                    color=METHOD_COLORS[method],
                    linestyle=line_styles[method],
                    linewidth=2,
                    label=f'{method} (AUC={roc_auc:.3f})')
        ax.plot([0,1],[0,1],'k--', linewidth=0.7)
        ax.set_title(f'mode={mode}')
        ax.set_xlabel('FPR')
        ax.set_ylabel('TPR')
        ax.legend(fontsize=9)
        ax.set_xlim([0,1])
        ax.set_ylim([0,1.02])

    plt.tight_layout()
    if save:
        path = os.path.join(FIGURES_DIR, f'roc_overlay_{dataset}_{emb}.pdf')
        fig.savefig(path, bbox_inches='tight')
        print(f'Saved → {path}')
    plt.show()

# Focus on long-form datasets where improvement is most important
for ds in LONG_FORM:
    plot_roc_overlay(all_df, dataset=ds)

## 5. Similarity Distribution Comparison — MaxSim vs Baseline

In [ ]:
def plot_sim_distribution_comparison(all_df, emb='bge-base', mode='no_thinking', save=True):
    """KDE comparison of correct/incorrect similarity distributions: baseline vs maxsim."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f'Similarity Distribution: Baseline vs MaxSim  |  mode={mode}  |  emb={emb}',
        fontweight='bold'
    )

    method_linestyles = {'baseline': '-', 'maxsim': '--'}

    for ax, ds in zip(axes.flat, DATASETS):
        sub = all_df[
            (all_df['dataset'] == ds) &
            (all_df['thinking_mode'] == mode) &
            (all_df['emb_model'] == emb) &
            (all_df['method'].isin(['baseline', 'maxsim']))
        ]
        if sub.empty:
            ax.set_visible(False)
            continue
        for method, ls in method_linestyles.items():
            mdata = sub[sub['method'] == method]
            if mdata.empty:
                continue
            for label, color in [(1, PALETTE['correct']), (0, PALETTE['incorrect'])]:
                vals = mdata[mdata['is_correct'] == label]['similarity']
                if len(vals) < 2:
                    continue
                lname = 'Correct' if label == 1 else 'Incorrect'
                sns.kdeplot(vals, ax=ax, fill=False, alpha=0.8,
                            color=color, linestyle=ls, linewidth=1.5,
                            label=f'{method}/{lname}')
        acc = sub[sub['method']=='baseline']['is_correct'].mean() * 100
        ax.set_title(f'{ds}  (acc={acc:.1f}%)')
        ax.set_xlabel('Similarity Score')
        ax.set_ylabel('Density')
        ax.legend(fontsize=7)

    plt.tight_layout()
    if save:
        path = os.path.join(FIGURES_DIR, f'dist_comparison_maxsim_{mode}_{emb}.pdf')
        fig.savefig(path, bbox_inches='tight')
        print(f'Saved → {path}')
    plt.show()

for mode in MODES:
    plot_sim_distribution_comparison(all_df, mode=mode)

## 6. Method Ranking Summary Table

In [ ]:
if not imp_df.empty:
    # Average AUC per method × task type
    imp_df['form'] = imp_df['dataset'].apply(
        lambda d: 'short_form' if d in SHORT_FORM else 'long_form'
    )
    summary_table = (
        imp_df.groupby(['method', 'form', 'emb_model'])['auc']
        .mean()
        .reset_index()
        .pivot_table(index=['method','emb_model'], columns='form', values='auc')
    )
    summary_table['overall'] = summary_table.mean(axis=1)
    summary_table = summary_table.sort_values('overall', ascending=False)

    display(
        summary_table.style
        .background_gradient(subset=['short_form','long_form','overall'],
                             cmap='RdYlGn', vmin=0.5, vmax=0.97)
        .format('{:.4f}')
        .set_caption('Mean AUC by Method × Task Type (averaged over datasets and thinking modes)')
    )

    # Highlight best method per dataset
    print('\n=== Best Method per Dataset × Mode × Emb ===')
    best = (
        imp_df.groupby(['dataset','thinking_mode','emb_model'])
        .apply(lambda g: g.loc[g['auc'].idxmax(), ['method','auc']])
        .reset_index()
    )
    best_pivot = best.pivot_table(index=['dataset','thinking_mode'],
                                  columns='emb_model', values='method', aggfunc='first')
    display(best_pivot)

## 7. ΔAUC Summary — Average Improvement per Long-form Task

In [ ]:
if not imp_df.empty and 'baseline' in imp_df['method'].values:
    baseline_auc = imp_df[imp_df['method']=='baseline'].set_index(
        ['dataset','thinking_mode','emb_model'])['auc']

    rows = []
    for method in ['poincare', 'maxsim']:
        mdf = imp_df[imp_df['method'] == method]
        for _, row in mdf.iterrows():
            key = (row['dataset'], row['thinking_mode'], row['emb_model'])
            if key in baseline_auc.index:
                delta = row['auc'] - baseline_auc[key]
                rows.append({
                    'method': method, 'dataset': row['dataset'],
                    'thinking_mode': row['thinking_mode'],
                    'emb_model': row['emb_model'],
                    'auc_baseline': baseline_auc[key],
                    'auc_method': row['auc'],
                    'delta_auc': delta,
                })

    delta_df = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('ΔAUC per Method (method − baseline), Focus: Long-form Tasks',
                 fontweight='bold')

    for ax, form_ds in zip(axes, [LONG_FORM, SHORT_FORM]):
        sub = delta_df[delta_df['dataset'].isin(form_ds)]
        if sub.empty:
            continue
        sns.boxplot(
            data=sub, x='dataset', y='delta_auc', hue='method',
            palette={'poincare': METHOD_COLORS['poincare'],
                     'maxsim': METHOD_COLORS['maxsim']},
            ax=ax, width=0.5
        )
        ax.axhline(0, color='black', linewidth=0.9, linestyle='--')
        form_name = 'Long-form' if form_ds == LONG_FORM else 'Short-form'
        ax.set_title(f'{form_name} datasets')
        ax.set_xlabel('Dataset')
        ax.set_ylabel('ΔAUC')
        ax.tick_params(axis='x', rotation=15)

    plt.tight_layout()
    path = os.path.join(FIGURES_DIR, 'delta_auc_boxplot.pdf')
    fig.savefig(path, bbox_inches='tight')
    print(f'Saved → {path}')
    plt.show()

    # Print mean ΔAUC per method per task type
    delta_df['form'] = delta_df['dataset'].apply(
        lambda d: 'short_form' if d in SHORT_FORM else 'long_form'
    )
    print('\nMean ΔAUC:')
    print(delta_df.groupby(['method','form'])['delta_auc'].mean().unstack().round(4))